In [1]:
import numpy as np
import networkx as nx
import torch

import sys
sys.path.append('../')
from models.staged import STAGED
from utils.graph_constructor import GraphConstructor
from tests.test_graph_constructor import create_square_grid_data as create_test_data
from utils.graph_data_handler import GraphDataHandler
from utils.visualization import visualize_attention_graph, visualize_graph

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /vast/palmer/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/torch_geometric/typing.py:110: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /vast/palmer/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [2]:
data = create_test_data()

In [3]:
# Define time lags as specified by the user
delta_gl = 1  # Time lag for gene -> ligand
delta_lr = 5  # Time lag for ligand -> receptor
delta_rg = 3  # Time lag for receptor -> gene
delta_gg = 7  # Time lag for gene -> gene
distance_threshold = 10

In [4]:
model = STAGED(
    num_genes=len(data['genes']),
    hidden_dim=64,
    num_gat_layers=1,
    num_mlp_layers=3,
    dropout=0.1,
    delta_gl=delta_gl,  # Time lag for gene -> ligand
    delta_lr=delta_lr,  # Time lag for ligand -> receptor 
    delta_rg=delta_rg,  # Time lag for receptor -> gene
    delta_gg=delta_gg,  # Time lag for gene -> gene
    add_self_loops=False,
)
graph_constructor = GraphConstructor(
    genes=data['genes'],
    ligand_receptor_pairs=data['ligand_receptor_pairs'],
    receptor_gene_pairs=data['receptor_gene_pairs'],
    cell_type_assignments=data['cell_type_assignments'],
    prior_grns=data['prior_grns']
)
graph_handler = GraphDataHandler(model)

In [5]:
time_point = 10
# Prepare cell graphs
cell_graphs = []
for cell_idx in range(data['n_cells']):
    # Construct and update graph for each cell
    base_graph = graph_constructor.construct_base_graph(cell_idx)
    updated_graph = graph_constructor.update_graph_with_neighbors(
        base_graph, cell_idx, data['cell_positions'], 
        time_point, distance_threshold=distance_threshold
    )
    # Assign features
    pyg_graph = graph_constructor.assign_node_features(
        updated_graph, cell_idx, time_point, 
        data['gene_expression'],
        delta_gl, delta_lr, delta_rg, delta_gg
    )
    cell_graphs.append(pyg_graph)

In [6]:
batch_size = 10
predictions, attention_weights, node_ptr = graph_handler.process_cell_graphs(
    cell_graphs,
    num_genes=len(data['genes']),
    batch_size=batch_size
)
results = predictions.detach()
attention_results = attention_weights


In [ ]:
results.shape

torch.Size([4, 6])

In [ ]:
cell_graphs

[Data(x=[16, 1], edge_index=[2, 18], gene_node_indices=[6], node_names=[16], node_types=[16]),
 Data(x=[16, 1], edge_index=[2, 18], gene_node_indices=[6], node_names=[16], node_types=[16]),
 Data(x=[16, 1], edge_index=[2, 18], gene_node_indices=[6], node_names=[16], node_types=[16]),
 Data(x=[16, 1], edge_index=[2, 18], gene_node_indices=[6], node_names=[16], node_types=[16])]

In [9]:
data['n_cells']

4

In [10]:
data['gene_expression'].shape

torch.Size([15, 4, 6])

In [11]:
data.keys()

dict_keys(['gene_expression', 'cell_positions', 'genes', 'cell_type_assignments', 'ligand_receptor_pairs', 'receptor_gene_pairs', 'prior_grns', 'n_time_points', 'n_cells', 'n_genes'])